## Importing dependencies

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import learning_curve, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import seaborn as sns

## Loading data and visualizating classes

In [ ]:
def check_class_balance(file_path: str):
    print("Loading data...")
    df = pd.read_csv(file_path)
    
    df = df.dropna(subset=['score', 'content'])
    
    # Function to convert stars to classes
    def map_sentiment(score):
        if score <= 2:
            return -1 
        elif score == 3:
            return 0  
        else:
            return 1  

    df['sentiment'] = df['score'].apply(map_sentiment)
    
    counts = df['sentiment'].value_counts()
    percentages = df['sentiment'].value_counts(normalize=True) * 100
    
    print("\n" + "="*30)
    print(f"TOTAL REVIEWS: {len(df)}")
    print("="*30)
    
    labels = {1: 'Positive (1)', -1: 'Negative (-1)', 0: 'Neutral (0)'}
    
    for sentiment_class in [1, -1, 0]:
        count = counts.get(sentiment_class, 0)
        pct = percentages.get(sentiment_class, 0)
        print(f"{labels[sentiment_class]:<15} | {count:>5} pcs. | {pct:>5.1f}%")
        
    print("="*30)
    
    # Visualization
    plt.figure(figsize=(8, 5))
    ax = counts.rename(labels).plot(
        kind='bar', 
        color=['#2ecc71', '#e74c3c', '#95a5a6'],
        edgecolor='black'
    )
    plt.title('Sentiment Class Balance', fontsize=14)
    plt.ylabel('Number of Reviews', fontsize=12)
    plt.xticks(rotation=0, fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    for p in ax.patches:
        ax.annotate(str(p.get_height()), (p.get_x() * 1.005, p.get_height() * 1.015))
        
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    check_class_balance("data/kyrgyz_only_lenient.csv")

## Removing neutral sentiment text

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'([!?.])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
        
    return text

def prepare_dataset(input_csv, output_csv):
    print("Loading data...")
    df = pd.read_csv(input_csv)
    
    initial_len = len(df)
    print(f"Initial size: {initial_len} rows")
    
    df = df[df['score'] != 3].copy()
    print(f"Removed neutral reviews: {initial_len - len(df)}")
    
    df['sentiment'] = df['score'].apply(lambda x: 1 if x > 3 else 0)
    
    print("Cleaning text (removing links, extra spaces, converting to lowercase)...")
    df['clean_text'] = df['content'].apply(clean_text)
    
    df = df[df['clean_text'] != ""]
    
    print("\nFinal sentiment distribution:")
    print(df['sentiment'].value_counts())
    print(df['sentiment'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
    
    df_final = df[['clean_text', 'sentiment', 'score']]
    df_final.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nReady data saved to file: {output_csv}")

if __name__ == "__main__":
    INPUT_FILE = 'data/kyrgyz_only_lenient.csv'
    OUTPUT_FILE = 'data/kyrgyz_ready_for_ml.csv'
    
    prepare_dataset(INPUT_FILE, OUTPUT_FILE)

## Class distribution

In [ ]:
def plot_class_distribution(csv_file):
    print(f"Loading data from {csv_file}...")
    
    try:
        df = pd.read_csv(csv_file)
    except FileNotFoundError:
        print(f"Error: File {csv_file} not found. Make sure you ran the previous script.")
        return

    if 'sentiment' not in df.columns:
        print("Error: No 'sentiment' column in the dataset.")
        return

    sns.set_theme(style="whitegrid")
    plt.rcParams.update({'font.size': 12})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Sentiment Class Distribution (0 - Negative, 1 - Positive)', fontsize=16, fontweight='bold')

    sns.countplot(data=df, x='sentiment', ax=axes[0], palette=['#ff6b6b', '#4ecdc4'])
    axes[0].set_title('Number of Reviews by Class')
    axes[0].set_xlabel('Sentiment Class')
    axes[0].set_ylabel('Count')
    
    for p in axes[0].patches:
        axes[0].annotate(f'{int(p.get_height())}', 
                         (p.get_x() + p.get_width() / 2., p.get_height()), 
                         ha='center', va='center', 
                         xytext=(0, 9), 
                         textcoords='offset points')

    class_counts = df['sentiment'].value_counts()
    axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', 
                startangle=90, colors=['#4ecdc4', '#ff6b6b'], shadow=True, explode=(0.05, 0))
    axes[1].set_title('Percentage Ratio')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    INPUT_FILE = 'data/kyrgyz_ready_for_ml.csv' 
    
    plot_class_distribution(INPUT_FILE)

## Baseline Model Training (TF-IDF + Logistic Regression) and Error Analysis

In [ ]:
def train_baseline(csv_file):
    print("1. Loading data...")
    df = pd.read_csv(csv_file)
    
    df = df.dropna(subset=['clean_text', 'sentiment'])
    
    X = df['clean_text']
    y = df['sentiment']
    
    print("2. Splitting into train/test...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"   Training set size: {len(X_train)}")
    print(f"   Testing set size: {len(X_test)}")
    
    print("3. Text vectorization (TF-IDF)...")
    vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2), 
        max_features=10000, 
        min_df=3            
    )
    
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    
    print("4. Training Logistic Regression model...")
    model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    model.fit(X_train_tfidf, y_train)
    
    print("5. Evaluating quality...")
    y_pred = model.predict(X_test_tfidf)
    
    print("\n" + "="*50)
    print("CLASSIFICATION REPORT:")
    print("="*50)
    print(classification_report(y_test, y_pred, target_names=['0 (Negative)', '1 (Positive)']))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Pred. Negative', 'Pred. Positive'],
                yticklabels=['Actual Negative', 'Actual Positive'])
    plt.title('Confusion Matrix (Baseline TF-IDF + LogReg)')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.show()
    
    return model, vectorizer, X_test, y_test, y_pred


if __name__ == "__main__":
    INPUT_FILE = 'data/kyrgyz_ready_for_ml.csv'
    
    model, vectorizer, X_test, y_test, y_pred = train_baseline(INPUT_FILE)

    print("\n" + "="*50)
    print("START ERROR ANALYSIS (REVIEWS LONGER THAN 5 WORDS)")
    print("="*50)

    results_df = pd.DataFrame({
        'text': X_test.astype(str),  
        'true_label': y_test,  
        'pred_label': y_pred   
    })

    results_df['word_count'] = results_df['text'].apply(lambda x: len(x.split()))

    MIN_WORDS = 5

    false_positives = results_df[(results_df['true_label'] == 0) & 
                                 (results_df['pred_label'] == 1) & 
                                 (results_df['word_count'] >= MIN_WORDS)]

    false_negatives = results_df[(results_df['true_label'] == 1) & 
                                 (results_df['pred_label'] == 0) & 
                                 (results_df['word_count'] >= MIN_WORDS)]

    print(f"False Positives (longer than {MIN_WORDS} words): {len(false_positives)}")
    print(f"False Negatives (longer than {MIN_WORDS} words): {len(false_negatives)}\n")

    num_examples_to_show = 10
    
    print("=== EXAMPLES OF FALSE POSITIVES (Actual NEGATIVE [0], but model predicted POSITIVE [1]) ===")
    if not false_positives.empty:
        sample_fp = false_positives.sample(min(num_examples_to_show, len(false_positives)), random_state=42)
        for idx, row in sample_fp.iterrows():
            print(f"Words: {row['word_count']} | Text: {row['text']}")
            print("-" * 60)
    else:
        print("No suitable False Positives found.")

    print("\n=== EXAMPLES OF FALSE NEGATIVES (Actual POSITIVE [1], but model predicted NEGATIVE [0]) ===")
    if not false_negatives.empty:
        sample_fn = false_negatives.sample(min(num_examples_to_show, len(false_negatives)), random_state=42)
        for idx, row in sample_fn.iterrows():
            print(f"Words: {row['word_count']} | Text: {row['text']}")
            print("-" * 60)
    else:
        print("No suitable False Negatives found.")

## Baseline Model Training (Character N-gram TF-IDF + Logistic Regression) and Error Analysis

In [ ]:
def train_baseline_char(csv_file):
    print("1. Loading data...")
    df = pd.read_csv(csv_file)
    
    df = df.dropna(subset=['clean_text', 'sentiment'])
    
    X = df['clean_text']
    y = df['sentiment']
    
    print("2. Splitting into train/test...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"   Training set size: {len(X_train)}")
    print(f"   Testing set size: {len(X_test)}")
    
    print("3. Text vectorization (Character TF-IDF)...")
 
    vectorizer = TfidfVectorizer(
        analyzer='char_wb', 
        ngram_range=(3, 5), 
        max_features=15000, 
        min_df=3            
    )
    
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    
    print("4. Training Logistic Regression model...")
    model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    model.fit(X_train_tfidf, y_train)
    
    print("5. Evaluating quality...")
    y_pred = model.predict(X_test_tfidf)
    
    print("\n" + "="*50)
    print("CLASSIFICATION REPORT:")
    print("="*50)
    print(classification_report(y_test, y_pred, target_names=['0 (Negative)', '1 (Positive)']))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Pred. Negative', 'Pred. Positive'],
                yticklabels=['Actual Negative', 'Actual Positive'])
    plt.title('Confusion Matrix (Char TF-IDF + LogReg)')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.show()
    
    return model, vectorizer, X_test, y_test, y_pred


if __name__ == "__main__":
    INPUT_FILE = 'data/kyrgyz_ready_for_ml.csv'
    
    model, vectorizer, X_test, y_test, y_pred = train_baseline_char(INPUT_FILE)

    print("\n" + "="*50)
    print("START ERROR ANALYSIS (REVIEWS LONGER THAN 5 WORDS)")
    print("="*50)

    results_df = pd.DataFrame({
        'text': X_test.astype(str),  
        'true_label': y_test,  
        'pred_label': y_pred   
    })

    results_df['word_count'] = results_df['text'].apply(lambda x: len(x.split()))

    MIN_WORDS = 5

    false_positives = results_df[(results_df['true_label'] == 0) & 
                                 (results_df['pred_label'] == 1) & 
                                 (results_df['word_count'] >= MIN_WORDS)]

    false_negatives = results_df[(results_df['true_label'] == 1) & 
                                 (results_df['pred_label'] == 0) & 
                                 (results_df['word_count'] >= MIN_WORDS)]

    print(f"False Positives (longer than {MIN_WORDS} words): {len(false_positives)}")
    print(f"False Negatives (longer than {MIN_WORDS} words): {len(false_negatives)}\n")

    num_examples_to_show = 10
    
    print("=== EXAMPLES OF FALSE POSITIVES (Actual NEGATIVE [0], but model predicted POSITIVE [1]) ===")
    if not false_positives.empty:
        sample_fp = false_positives.sample(min(num_examples_to_show, len(false_positives)), random_state=42)
        for idx, row in sample_fp.iterrows():
            print(f"Words: {row['word_count']} | Text: {row['text']}")
            print("-" * 60)
    else:
        print("No suitable False Positives found.")

    print("\n=== EXAMPLES OF FALSE NEGATIVES (Actual POSITIVE [1], but model predicted NEGATIVE [0]) ===")
    if not false_negatives.empty:
        sample_fn = false_negatives.sample(min(num_examples_to_show, len(false_negatives)), random_state=42)
        for idx, row in sample_fn.iterrows():
            print(f"Words: {row['word_count']} | Text: {row['text']}")
            print("-" * 60)
    else:
        print("No suitable False Negatives found.")

## Sampling the data to get Golden Test Set

In [ ]:
df = pd.read_csv('data/kyrgyz_ready_for_ml.csv')
df = df.dropna(subset=['clean_text', 'sentiment'])
X = df['clean_text']
y = df['sentiment']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

test_df = pd.DataFrame({'text': X_test, 'old_label': y_test})

golden_df = test_df.sample(n=300, random_state=42)

golden_df['true_human_label'] = ""

golden_df.to_excel('data/golden_test_set_300.xlsx', index=False)
print("File golden_test_set_300.xlsx successfully created! It contains 300 rows.")

## Learning curve

In [ ]:
def plot_learning_curve(csv_file):
    print("1. Loading data...")
    df = pd.read_csv(csv_file)
    df = df.dropna(subset=['clean_text', 'sentiment'])
    
    X = df['clean_text']
    y = df['sentiment']
    
    print(f"Total data for analysis: {len(X)} rows")

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=10000, min_df=3)),
        ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    train_sizes_fractions = np.linspace(0.1, 1.0, 10)

    print("2. Calculating learning curve (this will take a couple of minutes)...")
    train_sizes, train_scores, test_scores = learning_curve(
        estimator=pipeline,
        X=X,
        y=y,
        train_sizes=train_sizes_fractions,
        cv=cv,
        scoring='f1_macro', 
        n_jobs=-1 
    )

    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    print("3. Drawing and saving the plot...")
    
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    
    plt.title("Learning Curve: Logistic Regression (Macro F1-score)", fontsize=16, pad=15)
    plt.xlabel("Number of training examples", fontsize=12)
    plt.ylabel("Macro F1-score", fontsize=12)
    
    plt.plot(train_sizes, train_scores_mean, 'o-', color="#1f77b4", label="Training score")
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="#1f77b4")
    
    plt.plot(train_sizes, test_scores_mean, 'o-', color="#ff7f0e", label="Cross-validation score")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="#ff7f0e")

    plt.legend(loc="lower right", fontsize=11)
    plt.ylim(0.5, 1.01) 
    plt.tight_layout()
    
    plt.show()


if __name__ == "__main__":
    
    INPUT_FILE = 'data/kyrgyz_ready_for_ml.csv'
    plot_learning_curve(INPUT_FILE)

## Evaluation of Classical ML Baselines (TF-IDF with Naive Bayes, Logistic Regression, and Linear SVC)

In [ ]:
def prepare_data_and_features(csv_file):
    """
    Loads data, performs split and vectorization (TF-IDF).
    Returns ready matrices and label vectors for model training.
    """
    print("-> Loading and preparing data...")
    df = pd.read_csv(csv_file).dropna(subset=['clean_text', 'sentiment'])
    X = df['clean_text']
    y = df['sentiment']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print("-> Text vectorization (TF-IDF Word Level)...\n")
    vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=10000, min_df=3)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    return X_train_tfidf, X_test_tfidf, y_train, y_test


def run_naive_bayes(X_train, X_test, y_train, y_test):
    """
    Trains and evaluates the Multinomial Naive Bayes model.
    """
    print("="*60)
    print("MODEL 1: Multinomial Naive Bayes")
    print("="*60)
    
    model = MultinomialNB()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['0 (Negative)', '1 (Positive)']))
    print("\n")


def run_logistic_regression(X_train, X_test, y_train, y_test):
    """
    Trains and evaluates the Logistic Regression model.
    """
    print("="*60)
    print("MODEL 2: Logistic Regression")
    print("="*60)
    
    model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['0 (Negative)', '1 (Positive)']))
    print("\n")


def run_linear_svc(X_train, X_test, y_train, y_test):
    """
    Trains and evaluates the Linear SVC model.
    """
    print("="*60)
    print("MODEL 3: Linear SVC")
    print("="*60)
    
    model = LinearSVC(class_weight='balanced', random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['0 (Negative)', '1 (Positive)']))
    print("\n")


if __name__ == "__main__":
    INPUT_FILE = 'data/kyrgyz_ready_for_ml.csv'
    
    X_train_tfidf, X_test_tfidf, y_train, y_test = prepare_data_and_features(INPUT_FILE)
    
    run_naive_bayes(X_train_tfidf, X_test_tfidf, y_train, y_test)
    run_logistic_regression(X_train_tfidf, X_test_tfidf, y_train, y_test)
    run_linear_svc(X_train_tfidf, X_test_tfidf, y_train, y_test)

## Final Evaluation of Lexical Baselines on the Golden Standard Dataset

In [ ]:
def run_ultimate_golden_classic(train_csv, golden_excel):
    print("1. Loading raw data and extracting X_train...")
    df = pd.read_csv(train_csv).dropna(subset=['clean_text', 'sentiment'])
    X = df['clean_text']
    y = df['sentiment']

    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("2. Configuring vectorizers (Word and Char)...")
    vec_word = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=10000, min_df=3)
    vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=15000, min_df=3)
    
    X_train_word = vec_word.fit_transform(X_train)
    X_train_char = vec_char.fit_transform(X_train)

    print("\n3. Loading and cleaning the Golden Standard...")
    df_golden = pd.read_excel(golden_excel)
  
    df_golden = df_golden[~df_golden['true_human_label'].astype(str).str.lower().isin(['x', 'х', 'drop'])]
    
    X_golden = df_golden['text'].astype(str)
    y_golden_true = df_golden['true_human_label'].astype(int)
    
    X_golden_word = vec_word.transform(X_golden)
    X_golden_char = vec_char.transform(X_golden)

    print(f"   Final clean sample size: {len(df_golden)} rows\n")
    print("="*60)
    print("RESULTS ON THE GOLDEN STANDARD (GOLDEN SET)")
    print("="*60)

    # --- 1. Word-level LogReg ---
    print("\n[Model 1: Logistic Regression + Word TF-IDF (Baseline)]")
    model_lr_word = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    model_lr_word.fit(X_train_word, y_train)
    print(classification_report(y_golden_true, model_lr_word.predict(X_golden_word), target_names=['0 (Negative)', '1 (Positive)']))

    # --- 2. Char-level LogReg ---
    print("\n[Model 2: Logistic Regression + Char TF-IDF (Morphology)]")
    model_lr_char = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    model_lr_char.fit(X_train_char, y_train)
    print(classification_report(y_golden_true, model_lr_char.predict(X_golden_char), target_names=['0 (Negative)', '1 (Positive)']))

    # --- 3. Word-level Linear SVC ---
    print("\n[Model 3: Linear SVC + Word TF-IDF]")
    model_svc = LinearSVC(class_weight='balanced', random_state=42, max_iter=1000)
    model_svc.fit(X_train_word, y_train)
    print(classification_report(y_golden_true, model_svc.predict(X_golden_word), target_names=['0 (Negative)', '1 (Positive)']))

if __name__ == "__main__":
    TRAIN_FILE = 'data/kyrgyz_ready_for_ml.csv'
    GOLDEN_FILE = 'data/golden_test_set_300.xlsx'
    run_ultimate_golden_classic(TRAIN_FILE, GOLDEN_FILE)

In [ ]:
df_golden = pd.read_excel("data/golden_test_set_300.xlsx")
df_golden[df_golden["true_human_label"] == "X"].shape

## Dataset Summary Statistics Extraction (Table 1 Metrics)

In [ ]:
df = pd.read_csv('data/kyrgyz_ready_for_ml.csv').dropna(subset=['clean_text', 'sentiment'])

df['word_count'] = df['clean_text'].apply(lambda x: len(str(x).split()))

df_pos = df[df['sentiment'] == 1]
df_neg = df[df['sentiment'] == 0]

print("--- 1. Total Reviews ---")
print(f"Positive (Class 1): {len(df_pos)}")
print(f"Negative (Class 0): {len(df_neg)}")
print(f"Total: {len(df)}\n")

print("--- 2. Average Words per Review ---")
print(f"Positive (Class 1): {df_pos['word_count'].mean():.1f}")
print(f"Negative (Class 0): {df_neg['word_count'].mean():.1f}")
print(f"Total: {df['word_count'].mean():.1f}\n")

print("--- 3. Max Words in Review ---")
print(f"Positive (Class 1): {df_pos['word_count'].max()}")
print(f"Negative (Class 0): {df_neg['word_count'].max()}")
print(f"Total: {df['word_count'].max()}\n")

print("--- 4. Vocabulary Size (Unique tokens/words) ---")
total_vocab = set(' '.join(df['clean_text'].astype(str)).split())
print(f"Total: {len(total_vocab)}")

## Class-Conditional Vocabulary Size Calculation

In [ ]:
print("Loading dataset...")
df = pd.read_csv("data/kyrgyz_ready_for_ml.csv").dropna(
    subset=["clean_text", "sentiment"]
)

def count_unique_tokens(series):
    unique_words = set(series.str.split().explode())
    return len(unique_words)


pos_vocab = count_unique_tokens(df[df["sentiment"] == 1]["clean_text"])
neg_vocab = count_unique_tokens(df[df["sentiment"] == 0]["clean_text"])
total_vocab = count_unique_tokens(df["clean_text"])

print("-" * 50)
print(f"Vocabulary Size Positive (Class 1): {pos_vocab}")
print(f"Vocabulary Size Negative (Class 0): {neg_vocab}")
print(f"Total Vocabulary Size: {total_vocab}")
print("-" * 50)

if total_vocab != 15695:
    print(
        "Note: If Total slightly differs from 15695, it means another tokenizer was used previously (e.g. Scikit-learn or FastText)."
    )

## Semantic Overlap Analysis: Vocabulary Intersection and Top Shared Tokens

In [ ]:
import pandas as pd

print("-> Loading dataset...")
df = pd.read_csv("data/kyrgyz_ready_for_ml.csv").dropna(
    subset=["clean_text", "sentiment"]
)

pos_words = df[df["sentiment"] == 1]["clean_text"].str.split().explode()
neg_words = df[df["sentiment"] == 0]["clean_text"].str.split().explode()

pos_vocab = set(pos_words)
neg_vocab = set(neg_words)
total_vocab = set(df["clean_text"].str.split().explode())

print("-" * 50)
print(f"Vocabulary Size Positive (Class 1): {len(pos_vocab)}")
print(f"Vocabulary Size Negative (Class 0): {len(neg_vocab)}")
print(f"Total Vocabulary Size: {len(total_vocab)}")
print("-" * 50)

common_vocab = pos_vocab.intersection(neg_vocab)
overlap_ratio = len(common_vocab) / len(total_vocab)
print(f"Semantic Overlap (intersection): {len(common_vocab)} tokens ({overlap_ratio:.1%} of the entire vocabulary)")
print("-" * 50)

pos_counts = pos_words.value_counts()
neg_counts = neg_words.value_counts()

word_stats = pd.DataFrame({
    "pos_count": pos_counts,
    "neg_count": neg_counts
}).fillna(0)

common_stats = word_stats.loc[list(common_vocab)].copy()

common_stats["total_count"] = common_stats["pos_count"] + common_stats["neg_count"]

top_20_common = common_stats.sort_values(by="total_count", ascending=False).head(20)

print("TOP-20 most frequent words appearing in BOTH classes:")
print(top_20_common[["pos_count", "neg_count", "total_count"]].astype(int))